In [1]:
import torch
import matplotlib.pyplot as plt

from transformers import AutoVideoProcessor, AutoModel, AutoImageProcessor

import os
import torch
import numpy as np

from torchcodec.decoders import VideoDecoder
from IPython.display import Video
import pandas as pd

KeyboardInterrupt: 

In [ ]:
hf_repo = "facebook/vjepa2-vitl-fpc64-256"

model = AutoModel.from_pretrained(hf_repo)
processor = AutoVideoProcessor.from_pretrained(hf_repo)

In [ ]:
dataset_path = '/data/scratch/hmz574/scratch-aid/videos/training_test_videos/'
output_path = '/data/scratch/hmz574/scratch-aid/videos/training_test_videos_clips/'
file_list = os.listdir(dataset_path)

In [ ]:
!mkdir /data/scratch/hmz574/scratch-aid/videos/training_test_videos_clips/

In [ ]:
train_list = ['V'+ str(x + 1) + '.mp4' for x in range (32)]
train_list

In [ ]:
decoder.metadata

In [ ]:
frame_rate = 30.0
offset = int(frame_rate * 0.1) # seconds
duration = int(frame_rate * 2) #seconds
video_name = 'V20'

In [ ]:
decoder = VideoDecoder(dataset_path + video_name + '.mp4')

In [ ]:
annotations = pd.read_csv('Video_annotation_V1-V40.tsv', sep='\t', names = ['start','end','video'])

In [ ]:
#v1 = VideoDecoder(dataset_path + train_list[0])
annotations = annotations[annotations['video'] == video_name]
annotations

In [ ]:
embeddings = []
for start in annotations['start']:
    scratch_index = (start + offset, start + offset + duration)
    nonscratch_index = (start - offset - duration, start - offset)
    scratch = decoder[scratch_index[0]:scratch_index[1]]
    nonscratch = decoder[nonscratch_index[0]:nonscratch_index[1]]
    with torch.no_grad():
        video = processor(scratch, return_tensors="pt").to(model.device)
        video_embeddings = model.get_vision_features(**video)
        scratch_average_embedding = video_embeddings[0].mean(0)
    with torch.no_grad():
        video = processor(nonscratch, return_tensors="pt").to(model.device)
        video_embeddings = model.get_vision_features(**video)
        nonscratch_average_embedding = video_embeddings[0].mean(0)
    embeddings.append([scratch_average_embedding, nonscratch_average_embedding])


In [ ]:
import umap
from sklearn.preprocessing import StandardScaler
import seaborn as sns

In [ ]:
e = [x[0] for x in embeddings]
e.extend([x[1] for x in embeddings])
l = ['s' for x in embeddings]
l.extend(['n' for x in embeddings])

In [ ]:
batch = [str(x) for x in range(len(embeddings))]
batch.extend(batch)
batch

In [ ]:
reducer = umap.UMAP()

scaled_features = StandardScaler().fit_transform(e)
embedding = reducer.fit_transform(scaled_features)
embedding.shape

#ax = sns.scatterplot(x = embedding[:,0],y = embedding[:,1], s=12,hue=l)
ax = sns.scatterplot(x = embedding[:,0],y = embedding[:,1], s=12,hue=batch)
#ax = sns.scatterplot(x = embedding[:,0],y = embedding[:,1])

ax.legend(loc = 'center left', bbox_to_anchor = (1, 0.5), ncol = 1)

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pca = PCA(n_components=8)
pca_features = pca.fit_transform(e)
sns.scatterplot(x = pca_features[:,0],y = pca_features[:,1], hue = batch)

In [ ]:
embeddings[0]

In [ ]:
from matplotlib import rc
import matplotlib.animation as animation
rc('animation', html='jshtml')

In [ ]:
fig, ax = plt.subplots()

imgs = decoder[6909:6969][:,:,::2,::2]
imgs = imgs.permute(0,2,3,1) # Permuting to (Bx)HxWxC format
frames = [[ax.imshow(imgs[i])] for i in range(len(imgs))]

ani = animation.ArtistAnimation(fig, frames)
ani

In [ ]:
processor = AutoImageProcessor.from_pretrained('facebook/dinov2-small')
model = AutoModel.from_pretrained('facebook/dinov2-small')

In [ ]:
embeddings = []
for start in annotations['start']:
    scratch_index = (start + offset, start + offset + duration)
    nonscratch_index = (start - offset - duration - 1000, start - offset - 1000)
    scratch = decoder[scratch_index[0]:scratch_index[1]]
    nonscratch = decoder[nonscratch_index[0]:nonscratch_index[1]]
    with torch.no_grad():
        inputs = processor(images=scratch, return_tensors="pt")
        outputs = model(**inputs)
        last_hidden_state = outputs['last_hidden_state']
        scratch_cls_token = last_hidden_state[:,0]
    with torch.no_grad():
        inputs = processor(images=nonscratch, return_tensors="pt")
        outputs = model(**inputs)
        last_hidden_state = outputs['last_hidden_state']
        nonscratch_cls_token = last_hidden_state[:,0]
    embeddings.append([scratch_cls_token, nonscratch_cls_token])


In [ ]:
scratch = [x[0] for x in embeddings]
scratch = torch.concat(scratch, dim=0)
#scratch = torch.stack(scratch, dim=0)
nonscratch = [x[1] for x in embeddings]
nonscratch = torch.concat(nonscratch, dim=0)
#nonscratch = torch.stack(nonscratch, dim=0)
e = torch.concat([scratch, nonscratch], dim=0)
l = ['s' for x in scratch]
l.extend(['n' for x in nonscratch])


In [ ]:
reducer = umap.UMAP()

scaled_features = StandardScaler().fit_transform(e)
embedding = reducer.fit_transform(scaled_features)
embedding.shape

ax = sns.scatterplot(x = embedding[:,0],y = embedding[:,1], s=12,hue=l)
#ax = sns.scatterplot(x = embedding[:,0],y = embedding[:,1], s=12,hue=batch)
#ax = sns.scatterplot(x = embedding[:,0],y = embedding[:,1])

ax.legend(loc = 'center left', bbox_to_anchor = (1, 0.5), ncol = 1)

In [ ]:
pca = PCA(n_components=8)
pca_features = pca.fit_transform(e)
sns.scatterplot(x = pca_features[:,0],y = pca_features[:,1], hue = l, s=5)